In [5]:
# ============================================================
# Cell 1 — Imports & Setup
# ============================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import MultiLabelBinarizer

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
print("Libraries loaded ✅")

Libraries loaded ✅


In [ ]:
# ============================================================
# Cell 2 — Load All Datasets
# ============================================================

emp  = pd.read_csv('../../dataset/employees.csv')
tf   = pd.read_csv('../../dataset/team_formations.csv')
pr   = pd.read_csv('../../dataset/performance_reviews.csv')
ta   = pd.read_csv('../../dataset/task_assignments.csv')
wh   = pd.read_csv('../../dataset/workload_history.csv')
proj = pd.read_csv('../../dataset/projects.csv')
tasks = pd.read_csv('../../dataset/tasks.csv')

print("Shapes:")
print(f"  employees        : {emp.shape}")
print(f"  team_formations  : {tf.shape}")
print(f"  performance_rev  : {pr.shape}")
print(f"  task_assignments : {ta.shape}")
print(f"  workload_history : {wh.shape}")
print(f"  projects         : {proj.shape}")

Shapes:
  employees        : (2000, 42)
  team_formations  : (1500, 40)
  performance_rev  : (4000, 55)
  task_assignments : (30000, 29)
  workload_history : (110369, 50)
  projects         : (1000, 43)


In [8]:
# ============================================================
# Cell 3 — Clean employees.csv
# ============================================================

# --- Datetime parsing ---
emp['hire_date']    = pd.to_datetime(emp['hire_date'], format='mixed')
emp['last_updated'] = pd.to_datetime(emp['last_updated'], format='mixed')
emp['created_at']   = pd.to_datetime(emp['created_at'], format='mixed')

# Derive tenure in days from today
emp['tenure_days'] = (pd.Timestamp.today() - emp['hire_date']).dt.days

# --- Boolean fix (ensure actual bool, not string) ---
emp['is_available']               = emp['is_available'].astype(bool)
emp['cross_functional_experience'] = emp['cross_functional_experience'].astype(bool)
emp['mentoring_experience']        = emp['mentoring_experience'].astype(bool)

# --- Handle nulls ---
# certifications: 114 nulls -> fill with empty string (means no certification)
emp['certifications'] = emp['certifications'].fillna('')

# past_team_members: all null -> drop column (no usable data)
emp.drop(columns=['past_team_members'], inplace=True)

# --- Derived features ---
emp['success_rate'] = (
    emp['successful_project_count'] /
    (emp['successful_project_count'] + emp['failed_project_count'])
).fillna(0)

# --- Ordinal encoding ---
seniority_order = {'Junior': 1, 'Mid': 2, 'Senior': 3, 'Lead': 4, 'Principal': 5}
stress_order    = {'Low': 1, 'Medium': 2, 'High': 3}

emp['seniority_encoded'] = emp['seniority_level'].map(seniority_order)
emp['stress_encoded']    = emp['stress_level'].map(stress_order)

# --- Parse skills from comma-separated strings to lists ---
emp['primary_skills_list']   = emp['primary_skills'].fillna('').str.split(',').apply(
    lambda x: [s.strip() for s in x if s.strip()])
emp['secondary_skills_list'] = emp['secondary_skills'].fillna('').str.split(',').apply(
    lambda x: [s.strip() for s in x if s.strip()])
emp['all_skills_list']       = emp['primary_skills_list'] + emp['secondary_skills_list']

print("employees.csv cleaned ✅")
print(emp[['employee_id','seniority_encoded','stress_encoded','tenure_days','success_rate']].head(3))

employees.csv cleaned ✅
  employee_id  seniority_encoded  stress_encoded  tenure_days  success_rate
0      EMP001                  1               2          778      0.000000
1      EMP002                  2               3         2824      1.000000
2      EMP003                  1               3          617      0.833333


In [9]:
# ============================================================
# Cell 4 — Multi-hot Encode Skills
# ============================================================

mlb = MultiLabelBinarizer()
skill_matrix = mlb.fit_transform(emp['all_skills_list'])
skill_df     = pd.DataFrame(skill_matrix, columns=mlb.classes_, index=emp.index)

print(f"Skill matrix shape: {skill_df.shape}")
print(f"Total unique skills: {len(mlb.classes_)}")
print(skill_df.iloc[:3, :8])  # preview first 8 skill columns

Skill matrix shape: (2000, 63)
Total unique skills: 63
   A/B Testing  AWS  Adobe Suite  Adobe XD  Agile  Bloomberg  CRM  \
0            0    0            0         0      0          0    0   
1            0    0            1         0      0          0    0   
2            0    0            0         0      0          0    0   

   Communication  
0              0  
1              0  
2              0  


In [10]:
# ============================================================
# Cell 5 — Clean performance_reviews.csv
#           Keep only latest review per employee
# ============================================================

pr['review_date'] = pd.to_datetime(pr['review_date'])

# Sort and keep latest review per employee
pr_latest = (
    pr.sort_values('review_date', ascending=False)
      .groupby('employee_id', as_index=False)
      .first()
)

# Keep only useful columns for merging
pr_cols = [
    'employee_id',
    'overall_performance_score',
    'normalized_performance_score',
    'collaboration_score',
    'communication_score',
    'leadership_score',
    'problem_solving_score',
    'on_time_delivery_rate',
    'productivity_vs_peers',
    'promotion_recommended',
    'salary_increase_recommended'
]
pr_latest = pr_latest[pr_cols]

print("performance_reviews.csv cleaned ✅")
print(f"Unique employees with reviews: {pr_latest['employee_id'].nunique()}")
print(pr_latest.head(3))

performance_reviews.csv cleaned ✅
Unique employees with reviews: 1705
  employee_id  overall_performance_score  normalized_performance_score  \
0      EMP001                      85.50                         82.60   
1      EMP002                      78.10                         75.90   
2      EMP003                       4.14                         41.41   

   collaboration_score  communication_score  leadership_score  \
0                 5.10                 5.80              5.10   
1                 9.20                 7.80              3.60   
2                 6.62                 9.23              4.32   

   problem_solving_score  on_time_delivery_rate  productivity_vs_peers  \
0                   6.50                  76.70                  12.90   
1                   6.70                  97.90                  -3.80   
2                   8.78                  83.87                  12.57   

   promotion_recommended  salary_increase_recommended  
0                  

In [11]:
# ============================================================
# Cell 6 — Clean workload_history.csv
#           Use last 6 weeks only, aggregate per employee
# ============================================================

wh['date'] = pd.to_datetime(wh['date'])

# Filter to last 6 weeks
cutoff = wh['date'].max() - pd.Timedelta(weeks=6)
wh_recent = wh[wh['date'] >= cutoff].copy()

print(f"Workload records after cutoff ({cutoff.date()}): {len(wh_recent)}")

# Aggregate per employee
wh_agg = wh_recent.groupby('employee_id').agg(
    avg_hours_worked      = ('total_hours_worked',    'mean'),
    avg_overtime_hours    = ('overtime_hours',         'mean'),
    avg_workload_capacity = ('workload_vs_capacity',   'mean'),
    avg_burnout_risk      = ('burnout_risk_today',     'mean'),
    avg_productivity      = ('productivity_score',     'mean'),
    avg_stress_score      = ('stress_score',           'mean'),
    avg_focus_hours       = ('focused_work_hours',     'mean'),
    late_hours_frequency  = ('late_hours_indicator',   'sum'),
    weekend_work_count    = ('weekend_work_indicator', 'sum'),
).reset_index()

# Derived: available capacity estimate
# weekly_capacity_hours comes from employees.csv, merge later
print("workload_history.csv aggregated ✅")
print(wh_agg.head(3))

Workload records after cutoff (2027-05-28): 211
workload_history.csv aggregated ✅
  employee_id  avg_hours_worked  avg_overtime_hours  avg_workload_capacity  \
0      EMP013          8.182223            0.574702             110.091111   
1      EMP016          8.394144            0.000000             108.176287   
2     EMP0238          6.650306            0.000000             110.067305   

   avg_burnout_risk  avg_productivity  avg_stress_score  avg_focus_hours  \
0         37.787673         65.573467         35.658601         4.511117   
1         19.924532         92.405589         42.276203         3.733945   
2         34.450958         57.261870         13.681784         0.467972   

   late_hours_frequency  weekend_work_count  
0                     0                   0  
1                     0                   0  
2                     0                   0  


In [12]:
# ============================================================
# Cell 7 — Clean task_assignments.csv
#           Derive avg skill match & success rate per employee
# ============================================================

# Fix assignment_success — mixed bool/NaN
ta['assignment_success'] = ta['assignment_success'].map(
    {True: 1, False: 0, 'True': 1, 'False': 0}
)

ta_agg = ta.groupby('employee_id').agg(
    avg_skill_match       = ('skill_match_score',        'mean'),
    avg_suitability       = ('overall_suitability_score','mean'),
    avg_quality_rating    = ('quality_rating',            'mean'),
    avg_efficiency        = ('efficiency_score',          'mean'),
    avg_assignment_sat    = ('assignment_satisfaction',   'mean'),
    task_success_rate     = ('assignment_success',        'mean'),
    total_assignments     = ('assignment_id',             'count'),
).reset_index()

print("task_assignments.csv aggregated ✅")
print(ta_agg.head(3))

task_assignments.csv aggregated ✅
  employee_id  avg_skill_match  avg_suitability  avg_quality_rating  \
0      EMP001        73.986121        75.102735            8.010090   
1      EMP002        66.181747        73.193962            7.315987   
2      EMP003        69.240458        75.862821            8.161032   

   avg_efficiency  avg_assignment_sat  task_success_rate  total_assignments  
0        0.915878            7.557113           0.600000                 29  
1        0.931584            7.621277           0.666667                 27  
2        0.985192            7.284464           0.600000                 40  


In [13]:
# ============================================================
# Cell 8 — Build Master Employee Feature Table
# ============================================================

# Start with employees base
master = emp.copy()

# Join performance review features
master = master.merge(pr_latest, on='employee_id', how='left')

# Join workload aggregates
master = master.merge(wh_agg, on='employee_id', how='left')

# Join task assignment aggregates
master = master.merge(ta_agg, on='employee_id', how='left')

# Derived: available capacity hours
master['available_capacity'] = master['weekly_capacity_hours'] - (
    master['avg_hours_worked'].fillna(0)
)

print(f"Master table shape: {master.shape}")
print(f"Null counts (top 10):\n{master.isnull().sum().sort_values(ascending=False).head(10)}")

Master table shape: (2000, 75)
Null counts (top 10):
avg_burnout_risk         1800
late_hours_frequency     1800
avg_overtime_hours       1800
avg_workload_capacity    1800
avg_productivity         1800
avg_stress_score         1800
avg_focus_hours          1800
weekend_work_count       1800
avg_hours_worked         1800
on_time_delivery_rate     295
dtype: int64


In [14]:
# ============================================================
# Cell 9 — Normalize Numeric Columns
# ============================================================

numeric_cols = [
    'years_of_experience', 'technical_proficiency_score',
    'domain_expertise_score', 'historical_performance_score',
    'average_task_completion_rate', 'collaboration_score',
    'communication_effectiveness', 'leadership_potential',
    'burnout_risk_score', 'work_life_balance_score',
    'success_rate', 'tenure_days',
    'avg_hours_worked', 'avg_overtime_hours',
    'avg_workload_capacity', 'avg_burnout_risk',
    'avg_productivity', 'avg_skill_match',
    'avg_suitability', 'avg_efficiency',
    'available_capacity'
]

# Only normalize columns that exist and have no all-null
existing_cols = [c for c in numeric_cols if c in master.columns]

scaler = MinMaxScaler()
master[existing_cols] = scaler.fit_transform(master[existing_cols].fillna(0))

print("Normalization done ✅")
print(master[existing_cols].describe().round(3))

Normalization done ✅
       years_of_experience  technical_proficiency_score  \
count             2000.000                     2000.000   
mean                 0.213                        0.394   
std                  0.227                        0.193   
min                  0.000                        0.000   
25%                  0.031                        0.257   
50%                  0.128                        0.374   
75%                  0.324                        0.534   
max                  1.000                        1.000   

       domain_expertise_score  historical_performance_score  \
count                2000.000                      2000.000   
mean                    0.433                         0.511   
std                     0.204                         0.201   
min                     0.000                         0.000   
25%                     0.299                         0.363   
50%                     0.415                         0.506   
75%   

In [15]:
# ============================================================
# Cell 10 — Clean team_formations.csv
#            Parse member_ids, set target variable
# ============================================================

tf['formation_date']  = pd.to_datetime(tf['formation_date'])
tf['dissolution_date'] = pd.to_datetime(tf['dissolution_date'], errors='coerce')

# Parse member_ids string → list
tf['member_ids_list'] = tf['member_ids'].str.split(',').apply(
    lambda x: [s.strip() for s in x] if isinstance(x, list) else x
)

# Fix met_deadline — mixed bool/NaN
tf['met_deadline'] = tf['met_deadline'].map(
    {True: 1, False: 0, 'True': 1, 'False': 0}
)

# --- Target variable ---
# Using actual_performance_score as regression target
# OR binarize for classification (>=80 = success)
tf['team_success'] = (tf['actual_performance_score'] >= 80).astype(int)

# Time-based train/test split (no random split — avoids leakage)
split_date = tf['formation_date'].quantile(0.8)  # 80% train
tf_train = tf[tf['formation_date'] <= split_date]
tf_test  = tf[tf['formation_date'] >  split_date]

print(f"Team formations cleaned ✅")
print(f"Train size: {len(tf_train)} | Test size: {len(tf_test)}")
print(f"Target balance:\n{tf['team_success'].value_counts()}")

Team formations cleaned ✅
Train size: 1201 | Test size: 299
Target balance:
team_success
0    872
1    628
Name: count, dtype: int64


In [16]:
# ============================================================
# Cell 11 — Sanity Checks
# ============================================================

print("=== Duplicate employee_ids in master ===")
print(master['employee_id'].duplicated().sum())  # should be 0

print("\n=== Score range check (should all be 0–1 after normalization) ===")
print(master[existing_cols].min().min(), "to", master[existing_cols].max().max())

print("\n=== Skill encoding check (first employee) ===")
sample_emp = emp.iloc[0]
print(f"  Employee: {sample_emp['employee_id']}")
print(f"  Skills: {sample_emp['all_skills_list']}")
matching_cols = [c for c in skill_df.columns if skill_df.loc[0, c] == 1]
print(f"  Encoded ON columns: {matching_cols}")

print("\n=== Master table final shape ===")
print(master.shape)

print("\n✅ Data Prep Complete — Ready for Feature Engineering & Modeling")

=== Duplicate employee_ids in master ===
0

=== Score range check (should all be 0–1 after normalization) ===
0.0 to 1.0

=== Skill encoding check (first employee) ===
  Employee: EMP001
  Skills: ['Illustrator', 'Sketch', 'Presentation', 'Leadership']
  Encoded ON columns: ['Illustrator', 'Leadership', 'Presentation', 'Sketch']

=== Master table final shape ===
(2000, 75)

✅ Data Prep Complete — Ready for Feature Engineering & Modeling


In [18]:
# ============================================================
# Cell 12 — Team-Level Feature Extraction Helper
#            Aggregates member features into team vectors
# ============================================================
import json

def parse_json_col(val):
    """Safely parse JSON-like strings from role_distribution / seniority_mix"""
    try:
        return json.loads(val.replace("'", '"'))
    except:
        return {}

def extract_team_features(row, master_df):
    """
    Given a team row from team_formations.csv,
    look up each member in master_df and compute team-level aggregates.
    Returns a flat feature dict.
    """
    member_ids = [m.strip() for m in str(row['member_ids']).split(',')]
    members    = master_df[master_df['employee_id'].isin(member_ids)]

    if members.empty:
        return None

    feats = {}

    # ── Team size ──────────────────────────────────────────
    feats['team_size'] = len(members)

    # ── Skill diversity: unique skills across all members ──
    all_skills = []
    for s in members['all_skills_list']:
        if isinstance(s, list):
            all_skills.extend(s)
    feats['unique_skill_count']  = len(set(all_skills))
    feats['avg_skills_per_member'] = len(all_skills) / max(len(members), 1)

    # ── Performance aggregates ─────────────────────────────
    feats['avg_performance']      = members['historical_performance_score'].mean()
    feats['min_performance']      = members['historical_performance_score'].min()
    feats['std_performance']      = members['historical_performance_score'].std()

    # ── Collaboration & communication ──────────────────────
    feats['avg_collaboration']    = members['collaboration_score'].mean()
    feats['avg_communication']    = members['communication_effectiveness'].mean()

    # ── Leadership ─────────────────────────────────────────
    feats['avg_leadership']       = members['leadership_potential'].mean()
    feats['has_senior_or_above']  = int(members['seniority_encoded'].max() >= 3)

    # ── Workload & burnout ─────────────────────────────────
    feats['avg_burnout_risk']     = members['burnout_risk_score'].mean()
    feats['max_burnout_risk']     = members['burnout_risk_score'].max()
    feats['avg_available_cap']    = members['available_capacity'].mean()
    feats['avg_workload_capacity']= members['avg_workload_capacity'].mean() \
                                    if 'avg_workload_capacity' in members.columns else np.nan

    # ── Experience ─────────────────────────────────────────
    feats['avg_experience']       = members['years_of_experience'].mean()
    feats['experience_range']     = members['years_of_experience'].max() \
                                  - members['years_of_experience'].min()

    # ── Technical proficiency ──────────────────────────────
    feats['avg_technical_score']  = members['technical_proficiency_score'].mean()
    feats['avg_domain_score']     = members['domain_expertise_score'].mean()

    # ── Task assignment quality ────────────────────────────
    feats['avg_skill_match']      = members['avg_skill_match'].mean() \
                                    if 'avg_skill_match' in members.columns else np.nan
    feats['avg_task_success_rate']= members['task_success_rate'].mean() \
                                    if 'task_success_rate' in members.columns else np.nan

    # ── Seniority mix diversity ────────────────────────────
    seniority_mix = parse_json_col(row['seniority_mix'])
    feats['seniority_levels_count'] = len(seniority_mix)   # how many distinct levels
    feats['junior_ratio'] = seniority_mix.get('Junior', 0) / max(feats['team_size'], 1)
    feats['senior_ratio'] = (
        seniority_mix.get('Senior', 0) +
        seniority_mix.get('Lead',   0) +
        seniority_mix.get('Principal', 0)
    ) / max(feats['team_size'], 1)

    # ── Cross-functional & mentoring ──────────────────────
    feats['cross_functional_ratio'] = members['cross_functional_experience'].mean()
    feats['mentoring_ratio']        = members['mentoring_experience'].mean()

    # ── Project success history ────────────────────────────
    feats['avg_member_success_rate'] = members['success_rate'].mean() \
                                       if 'success_rate' in members.columns else np.nan

    # ── Stress load ────────────────────────────────────────
    feats['avg_stress']     = members['stress_encoded'].mean()
    feats['high_stress_ratio'] = (members['stress_encoded'] == 3).mean()

    return feats

print("Helper functions defined ✅")

Helper functions defined ✅


In [19]:
# ============================================================
# Cell 13 — Apply Feature Extraction to All Teams
# ============================================================

team_feature_rows = []

for _, row in tf.iterrows():
    feats = extract_team_features(row, master)
    if feats is None:
        continue

    # Attach team identifiers and targets
    feats['team_id']                  = row['team_id']
    feats['project_id']               = row['project_id']
    feats['formation_date']           = row['formation_date']

    # Existing team-level scores (from dataset itself)
    feats['skill_diversity_score']    = row['skill_diversity_score']
    feats['experience_balance_score'] = row['experience_balance_score']
    feats['collaborative_history']    = row['collaborative_history_score']
    feats['communication_compat']     = row['communication_compatibility']
    feats['workload_balance_score']   = row['workload_balance_score']
    feats['timezone_compatibility']   = row['timezone_compatibility']

    # Targets
    feats['actual_performance_score'] = row['actual_performance_score']
    feats['met_deadline']             = row['met_deadline']
    feats['quality_rating']           = row['quality_rating']
    feats['team_success']             = int(row['actual_performance_score'] >= 80) \
                                        if pd.notna(row['actual_performance_score']) else np.nan

    team_feature_rows.append(feats)

team_features_df = pd.DataFrame(team_feature_rows)

print(f"Team feature matrix shape: {team_features_df.shape}")
print(f"Nulls per column:\n{team_features_df.isnull().sum().sort_values(ascending=False).head(10)}")
print(team_features_df.head(3))

KeyError: 'collaboration_score'

In [ ]:
# ============================================================
# Cell 14 — Project-Level Features
#            Attach project complexity/requirements to teams
# ============================================================

complexity_order = {'Low': 1, 'Medium': 2, 'High': 3, 'Very High': 4}
proj['complexity_encoded'] = proj['complexity_level'].map(complexity_order)

# Parse required_skills for projects
proj['required_skills_list'] = proj['required_skills'].fillna('').str.split(',').apply(
    lambda x: [s.strip() for s in x if s.strip()]
)

# Skill coverage: what % of required skills does the team collectively cover
def skill_coverage(team_row, proj_df, skill_mlb):
    """
    For a team row, compute what fraction of the project's
    required skills are covered by the team's combined skill set.
    """
    pid = team_row['project_id']
    project = proj_df[proj_df['project_id'] == pid]
    if project.empty:
        return np.nan

    required = set(project.iloc[0]['required_skills_list'])
    if not required:
        return np.nan

    # Get team member skills from team_formations member_ids
    member_ids = [m.strip() for m in str(
        tf.loc[tf['project_id'] == pid, 'member_ids'].iloc[0]
    ).split(',')]
    members = master[master['employee_id'].isin(member_ids)]

    team_skills = set()
    for s in members['all_skills_list']:
        if isinstance(s, list):
            team_skills.update(s)

    covered = required & team_skills
    return len(covered) / len(required)

# Merge project features into team_features_df
proj_subset = proj[[
    'project_id', 'complexity_encoded', 'priority',
    'budget', 'team_size', 'success_probability',
    'delay_risk_score', 'budget_overrun_risk',
    'quality_risk_score', 'strategic_importance'
]].rename(columns={'team_size': 'proj_required_team_size'})

# Encode priority
priority_order = {'Low': 1, 'Medium': 2, 'High': 3, 'Critical': 4}
proj_subset['priority_encoded'] = proj_subset['priority'].map(priority_order).fillna(2)
proj_subset.drop(columns=['priority'], inplace=True)

team_features_df = team_features_df.merge(proj_subset, on='project_id', how='left')

# Compute skill coverage per team
team_features_df['skill_coverage'] = team_features_df.apply(
    lambda row: skill_coverage(row, proj, mlb), axis=1
)

print("Project features merged ✅")
print(f"Updated shape: {team_features_df.shape}")
print(team_features_df[['team_id','skill_coverage','complexity_encoded','success_probability']].head(5))

In [ ]:
# ============================================================
# Cell 15 — Final Feature Matrix
#            Drop targets/IDs, finalize for modeling
# ============================================================

# Columns that are targets or identifiers — not features
drop_cols = [
    'team_id', 'project_id', 'formation_date',
    'actual_performance_score',   # regression target
    'met_deadline',               # alt target
    'quality_rating',             # alt target
    'team_success'                # classification target — keep separate
]

# Separate targets
y_regression     = team_features_df['actual_performance_score']
y_classification = team_features_df['team_success']
y_deadline       = team_features_df['met_deadline']

# Feature matrix
X = team_features_df.drop(columns=drop_cols, errors='ignore')

# Fill any remaining nulls with column median
X = X.fillna(X.median(numeric_only=True))

# Normalize new numeric columns added from project merge
new_num_cols = [
    'budget', 'success_probability', 'delay_risk_score',
    'budget_overrun_risk', 'quality_risk_score', 'strategic_importance'
]
existing_new = [c for c in new_num_cols if c in X.columns]
X[existing_new] = MinMaxScaler().fit_transform(X[existing_new])

print("=== Final Feature Matrix ===")
print(f"X shape       : {X.shape}")
print(f"Features      : {X.columns.tolist()}")
print(f"\nTarget shapes :")
print(f"  y_regression     : {y_regression.shape}  | nulls: {y_regression.isnull().sum()}")
print(f"  y_classification : {y_classification.shape} | nulls: {y_classification.isnull().sum()}")
print(f"  y_deadline       : {y_deadline.shape}  | nulls: {y_deadline.isnull().sum()}")

In [ ]:
# ============================================================
# Cell 16 — Time-Based Train/Test Split & Save
# ============================================================

# Re-attach formation_date for splitting
X['formation_date'] = team_features_df['formation_date'].values

split_date = pd.to_datetime(X['formation_date']).quantile(0.8)

train_mask = pd.to_datetime(X['formation_date']) <= split_date
test_mask  = ~train_mask

X_train = X[train_mask].drop(columns=['formation_date'])
X_test  = X[test_mask].drop(columns=['formation_date'])

y_train_reg  = y_regression[train_mask]
y_test_reg   = y_regression[test_mask]

y_train_cls  = y_classification[train_mask]
y_test_cls   = y_classification[test_mask]

print(f"Train size : {X_train.shape}")
print(f"Test size  : {X_test.shape}")
print(f"\nClass balance (train):\n{y_train_cls.value_counts()}")
print(f"\nClass balance (test):\n{y_test_cls.value_counts()}")

# Save for use in modeling notebook
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv',  index=False)
y_train_reg.to_csv('y_train_regression.csv', index=False)
y_test_reg.to_csv('y_test_regression.csv',   index=False)
y_train_cls.to_csv('y_train_classification.csv', index=False)
y_test_cls.to_csv('y_test_classification.csv',   index=False)

print("\n✅ Feature extraction complete — CSVs saved, ready for modeling")